# Smart Contract Vulnerability Detection - Full Comparison
Training with all optimization thresholds: before_optimized, optimized_80p, optimized_50p, optimized_20p

Each model is trained from scratch with fresh parameters loaded from `MODEL_NAME`.

---
**Edit the constants in the cell below to configure the run.**

In [ ]:
# ============================================================
# CELL 0 — GLOBAL CONSTANTS (edit here to configure the run)
# ============================================================

# HuggingFace model hub ID used for tokeniser + base weights
MODEL_NAME = "microsoft/codebert-base"

# HuggingFace dataset hub ID  (must expose train / test splits)
DATASET_NAME = "JakeClark38a/smart-contract-vulnerability-detection"

# Text columns to iterate over during training
TEXT_COLUMNS = ["before_optimized", "optimized_80p", "optimized_50p", "optimized_20p"]

# Multi-label target columns
LABEL_COLUMNS = [
    "Arithmetic",
    "Unchecked Return Values For Low Level Calls",
    "Denial of Service",
    "Time manipulation",
    "Reentrancy",
]

# ---------- Training hyper-parameters ----------
NUM_EPOCHS          = 10
TRAIN_BATCH_SIZE    = 16
EVAL_BATCH_SIZE     = 32
LEARNING_RATE       = 2e-5
WEIGHT_DECAY        = 0.01
MAX_TOKEN_LENGTH    = 512   # None → use model max

# ---------- Output ----------
# Where Trainer saves checkpoints (and where we look for them to resume)
OUTPUT_BASE = "/kaggle/working/output"

# ---------- HuggingFace auth (optional — set if dataset/model is private) ----------
# Leave as None to use the environment variable HF_TOKEN automatically.
HF_TOKEN = None   # e.g. "hf_xxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxx"

print("Global constants loaded.")
print(f"  MODEL_NAME   : {MODEL_NAME}")
print(f"  DATASET_NAME : {DATASET_NAME}")
print(f"  TEXT_COLUMNS : {TEXT_COLUMNS}")
print(f"  OUTPUT_BASE  : {OUTPUT_BASE}")

In [ ]:
import os, sys, time, subprocess
import torch

print('='*60)
print('ENVIRONMENT CHECK')
print('='*60)
print(f'Python: {sys.version}')
print(f'PyTorch: {torch.__version__}')
print(f'CUDA available: {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)}')
print()

In [ ]:
print('='*60)
print('STEP 1: Install Dependencies')
print('='*60)

result = subprocess.run(
    [sys.executable, '-m', 'pip', 'install', 'uv', '-q'],
    capture_output=True, text=True, timeout=60
)
print(f'uv install: OK' if result.returncode == 0 else 'FAILED')

missing_packages = ['transformers', 'accelerate', 'datasets', 'scikit-learn', 'huggingface_hub']
result = subprocess.run(
    ['uv', 'pip', 'install', '--system'] + missing_packages,
    capture_output=True, text=True, timeout=600
)
print(f'Packages install: OK' if result.returncode == 0 else 'FAILED')

for pkg in ['transformers', 'datasets', 'pandas', 'scikit-learn']:
    try:
        mod = __import__(pkg)
        print(f'  [OK] {pkg}')
    except ImportError:
        print(f'  [MISSING] {pkg}')
print()

In [ ]:
print('='*60)
print('STEP 2: Load Dataset from HuggingFace Hub')
print('='*60)

import pandas as pd
from datasets import load_dataset
from huggingface_hub import login

# Authenticate if a token was provided in the constants cell
if HF_TOKEN:
    login(token=HF_TOKEN, add_to_git_credential=False)
    print('Logged in to HuggingFace Hub.')
else:
    print('No HF_TOKEN set — using public access / env HF_TOKEN.')

print(f'\nLoading dataset: {DATASET_NAME}')
hf_dataset = load_dataset(DATASET_NAME)

# Convert to pandas DataFrames
train_df = hf_dataset['train'].to_pandas()
test_df  = hf_dataset['test'].to_pandas()

print(f'Train samples : {len(train_df)}')
print(f'Test samples  : {len(test_df)}')
print(f'Columns       : {train_df.columns.tolist()}')
print(f'Labels        : {LABEL_COLUMNS}')
print()

In [ ]:
print('='*60)
print('STEP 3: Define Helper Functions')
print('='*60)

from transformers import AutoTokenizer, AutoModelForSequenceClassification
from transformers import TrainingArguments, Trainer
import numpy as np
from sklearn.metrics import precision_recall_fscore_support, hamming_loss, classification_report
import torch
from torch.utils.data import Dataset as TorchDataset
from pathlib import Path
import gc


def hamming_score(y_true, y_pred):
    acc_list = []
    for i in range(y_true.shape[0]):
        set_true = set(np.where(y_true[i])[0])
        set_pred = set(np.where(y_pred[i])[0])
        if len(set_true) == 0 and len(set_pred) == 0:
            acc_list.append(1.0)
        else:
            acc_list.append(
                len(set_true & set_pred) / float(len(set_true | set_pred))
            )
    return np.mean(acc_list)


def compute_metrics(eval_pred):
    logits, labels = eval_pred
    predictions = (logits > 0).astype(int)
    precision, recall, f1, _ = precision_recall_fscore_support(
        labels, predictions, average='weighted', zero_division=0
    )
    return {
        'precision':     precision,
        'recall':        recall,
        'f1':            f1,
        'hamming_score': hamming_score(labels, predictions),
        'hamming_loss':  hamming_loss(labels, predictions),
    }


class VulnerabilityDataset(TorchDataset):
    def __init__(self, encodings, labels):
        self.encodings = encodings
        self.labels    = labels

    def __getitem__(self, idx):
        item = {key: val[idx] for key, val in self.encodings.items()}
        item['labels'] = torch.tensor(self.labels[idx], dtype=torch.float)
        return item

    def __len__(self):
        return len(self.labels)


def _latest_checkpoint(directory: str):
    """Return the path of the most-recent Trainer checkpoint in *directory*,
    or None if no checkpoint exists."""
    ckpt_dir = Path(directory)
    if not ckpt_dir.exists():
        return None
    checkpoints = sorted(
        [p for p in ckpt_dir.iterdir() if p.name.startswith('checkpoint-')],
        key=lambda p: int(p.name.split('-')[-1]),
    )
    return str(checkpoints[-1]) if checkpoints else None


def train_and_evaluate(column_name, train_df, test_df, output_dir):
    """Train CodeBERT on *column_name* features and return result dict.
    Automatically resumes from the latest checkpoint when one exists."""
    print(f'\n' + '='*60)
    print(f'Training with column: {column_name}')
    print('='*60)

    col_output_dir = os.path.join(output_dir, column_name)
    Path(col_output_dir).mkdir(parents=True, exist_ok=True)

    # ── Check for an existing checkpoint to resume from ──────────────
    resume_from = _latest_checkpoint(col_output_dir)
    if resume_from:
        print(f'[RESUME] Found checkpoint: {resume_from}')
    else:
        print('[START] No checkpoint found — training from scratch.')

    # ── Data preparation ─────────────────────────────────────────────
    train_texts  = train_df[column_name].fillna('').astype(str).tolist()
    test_texts   = test_df[column_name].fillna('').astype(str).tolist()
    train_labels = train_df[LABEL_COLUMNS].values
    test_labels  = test_df[LABEL_COLUMNS].values

    print(f'Train: {len(train_texts)}, Test: {len(test_texts)}')

    tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

    train_encodings = tokenizer(
        train_texts, truncation=True, padding=True,
        max_length=MAX_TOKEN_LENGTH, return_tensors='pt'
    )
    test_encodings = tokenizer(
        test_texts, truncation=True, padding=True,
        max_length=MAX_TOKEN_LENGTH, return_tensors='pt'
    )

    train_dataset = VulnerabilityDataset(train_encodings, train_labels)
    test_dataset  = VulnerabilityDataset(test_encodings,  test_labels)

    train_size = int(0.8 * len(train_dataset))
    eval_size  = len(train_dataset) - train_size
    train_split, eval_dataset = torch.utils.data.random_split(
        train_dataset, [train_size, eval_size]
    )

    # ── Model ────────────────────────────────────────────────────────
    model = AutoModelForSequenceClassification.from_pretrained(
        MODEL_NAME,
        num_labels=len(LABEL_COLUMNS),
        problem_type='multi_label_classification',
    )
    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    model  = model.to(device)

    # ── Training args ────────────────────────────────────────────────
    training_args = TrainingArguments(
        output_dir=col_output_dir,
        num_train_epochs=NUM_EPOCHS,
        per_device_train_batch_size=TRAIN_BATCH_SIZE,
        per_device_eval_batch_size=EVAL_BATCH_SIZE,
        learning_rate=LEARNING_RATE,
        weight_decay=WEIGHT_DECAY,
        eval_strategy='epoch',
        save_strategy='epoch',
        load_best_model_at_end=False,
        save_total_limit=2,              # keep last 2 checkpoints for safety
        fp16=torch.cuda.is_available(),
        logging_steps=100,
        report_to='none',
    )

    trainer = Trainer(
        model=model,
        args=training_args,
        train_dataset=train_split,
        eval_dataset=eval_dataset,
        compute_metrics=compute_metrics,
    )

    # ── Train (resume_from_checkpoint handles partial runs) ──────────
    train_start = time.time()
    print('Training...')
    trainer.train(resume_from_checkpoint=resume_from)
    train_time = time.time() - train_start

    # ── Evaluation ───────────────────────────────────────────────────
    print('Evaluating on eval split...')
    test_start   = time.time()
    eval_results = trainer.evaluate()
    test_inference_time = time.time() - test_start

    print('Evaluating on held-out test set...')
    train_eval_start   = time.time()
    train_eval_results = trainer.evaluate(test_dataset)
    train_inference_time = time.time() - train_eval_start

    print('Generating classification report...')
    preds       = trainer.predict(test_dataset)
    pred_labels = (preds.predictions > 0).astype(int)
    print('\nClassification Report by Label:')
    print(classification_report(
        test_labels, pred_labels, target_names=LABEL_COLUMNS, zero_division=0
    ))

    # ── Save final model ─────────────────────────────────────────────
    model.save_pretrained(col_output_dir)
    tokenizer.save_pretrained(col_output_dir)
    print(f'Model saved to: {col_output_dir}')

    del model, trainer, train_dataset, test_dataset
    gc.collect()
    torch.cuda.empty_cache()

    return {
        'column':               column_name,
        'train_time':           train_time,
        'train_inference_time': train_inference_time,
        'test_inference_time':  test_inference_time,
        'num_train_samples':    len(train_texts),
        'num_test_samples':     len(test_texts),
        'precision':            eval_results.get('eval_precision',     0),
        'recall':               eval_results.get('eval_recall',        0),
        'f1':                   eval_results.get('eval_f1',            0),
        'hamming_score':        eval_results.get('eval_hamming_score', 0),
        'hamming_loss':         eval_results.get('eval_hamming_loss',  0),
    }


print('Helper functions defined.')
print()

In [ ]:
print('='*60)
print('STEP 4: Run All Experiments')
print('='*60)

from pathlib import Path
import json

Path(OUTPUT_BASE).mkdir(parents=True, exist_ok=True)

all_results = []
total_start = time.time()

for col in TEXT_COLUMNS:
    result = train_and_evaluate(
        column_name=col,
        train_df=train_df,
        test_df=test_df,
        output_dir=OUTPUT_BASE,
    )
    all_results.append(result)

    print(f'\nResults for {col}:')
    print(f'  Train Samples      : {result["num_train_samples"]}')
    print(f'  Test Samples       : {result["num_test_samples"]}')
    print(f'  Train Time         : {result["train_time"]/60:.1f} min')
    print(f'  Train Infer. Time  : {result["train_inference_time"]:.2f}s')
    print(f'  Test Infer. Time   : {result["test_inference_time"]:.2f}s')
    print(f'  Precision          : {result["precision"]:.4f}')
    print(f'  Recall             : {result["recall"]:.4f}')
    print(f'  F1                 : {result["f1"]:.4f}')
    print(f'  Hamming Score      : {result["hamming_score"]:.4f}')
    print(f'  Hamming Loss       : {result["hamming_loss"]:.4f}')

    gc.collect()
    torch.cuda.empty_cache()

total_time = time.time() - total_start
print(f'\nTotal training time: {total_time/60:.1f} minutes')
print()

In [ ]:
print('='*60)
print('STEP 5: Summary Comparison')
print('='*60)

import pandas as pd
import json

comparison_df = pd.DataFrame(all_results)
comparison_df = comparison_df[[
    'column', 'num_train_samples', 'num_test_samples',
    'train_time', 'train_inference_time', 'test_inference_time',
    'precision', 'recall', 'f1', 'hamming_score', 'hamming_loss'
]]
comparison_df.columns = [
    'Dataset', 'Train Samples', 'Test Samples',
    'Train Time (s)', 'Train Inference (s)', 'Test Inference (s)',
    'Precision', 'Recall', 'F1', 'Hamming Score', 'Hamming Loss'
]

print('\n' + '='*100)
print('FINAL RESULTS COMPARISON')
print('='*100)
print(comparison_df.to_string(index=False))
print('='*100)

comparison_csv = os.path.join(OUTPUT_BASE, 'comparison_results.csv')
comparison_df.to_csv(comparison_csv, index=False)
print(f'\nResults saved to: {comparison_csv}')

results_json = {
    'configuration': {
        'model':        MODEL_NAME,
        'dataset':      DATASET_NAME,
        'text_columns': TEXT_COLUMNS,
        'labels':       LABEL_COLUMNS,
        'num_epochs':   NUM_EPOCHS,
        'learning_rate': LEARNING_RATE,
    },
    'results':    all_results,
    'total_time': total_time,
}

results_json_path = os.path.join(OUTPUT_BASE, 'experiment_results.json')
with open(results_json_path, 'w') as f:
    json.dump(results_json, f, indent=2)
print(f'Results saved to: {results_json_path}')

print('\nAll experiments completed!')
print()